# Tema: Calidad y expectations

## Objetivos
Comparar observar, descartar y fallar; recuperar registros rechazados.

## Conceptos importantes para el examen
EXPECT registra métricas; ON VIOLATION DROP ROW descarta; ON VIOLATION FAIL UPDATE falla el flujo afectado. Una regla de calidad debe tratar nulos explícitamente.

**Dificultad:** Intermedio · **Tiempo estimado:** 75 min.

Añade resources/pipelines/quality.sql a un pipeline con lab.source_table y el destino impresos. Primero ejecuta sin quality_fail.sql. Añade ese segundo archivo solo para provocar el fallo esperado. La comparación local se ejecuta sin Lakeflow.

Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_23_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
pipeline_input = spark.createDataFrame([(i, i % 4, float(i*10) if i != 12 else -5.0) for i in range(1,13)], "order_id INT, customer_id INT, amount DOUBLE")
pipeline_input.write.format("delta").mode("overwrite").saveAsTable("pipeline_input")
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_input"
PIPELINE_SCHEMA = SCHEMA + "_pipeline"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(PIPELINE_SCHEMA)}")
print("lab.source_table =", SOURCE_TABLE)
print("Catálogo destino =", CATALOG, "Schema destino =", PIPELINE_SCHEMA)
RUN_PIPELINE_CHECKS = False  # Cambiar a True después de ejecutar el pipeline real.

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Perfilado batch

In [ ]:
display(pipeline_input.agg(F.count("*").alias("total"), F.sum(F.when(F.col("amount").isNull() | (F.col("amount")<0),1).otherwise(0)).alias("invalid")))

### 2. Separar cuarentena

In [ ]:
good = pipeline_input.filter("amount IS NOT NULL AND amount>=0")
bad = pipeline_input.filter("amount IS NULL OR amount<0")
bad.write.format("delta").mode("overwrite").saveAsTable("quality_quarantine_local")
assert good.count()+bad.count()==12

### 3. Verificar políticas del pipeline
Ejecuta quality.sql y abre la pestaña de calidad de cada dataset.

In [ ]:
if RUN_PIPELINE_CHECKS:
    prefix = f"{ident(CATALOG)}.{ident(PIPELINE_SCHEMA)}"
    assert spark.table(prefix + ".quality_observe").count()==12
    assert spark.table(prefix + ".quality_drop").count()==11
    assert spark.table(prefix + ".quality_quarantine").count()==1
else:
    print("Simulación local: observar conserva 12; descartar deja 11; fallar rechaza el lote.")

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Define dos reglas de negocio: clave obligatoria e importe no negativo. Evalúa cada una en PySpark.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Ejecuta quality.sql y compara EXPECT con DROP. Registra filas retenidas y métricas.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Añade quality_fail.sql al pipeline y provoca un fallo esperado. Identifica el flujo afectado, sin asumir rollback de todos los datasets.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Corrige el importe negativo en una fuente nueva y ejecuta Full refresh en un pipeline de prácticas apuntando a ella.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Crea un informe de tasa de rechazo y define un umbral que rechace un lote si supera 5%.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** No dependas solo de comparaciones con NULL.

**Pista 2:** EXPECT no elimina el negativo.

**Pista 3:** FAIL UPDATE rechaza el flujo con la violación; revisa el event log.

**Pista 4:** Un UPDATE de una fuente append-only no es un nuevo append válido.

**Pista 5:** 1/12 es mayor que 5%.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
rules = {"valid_key": "order_id IS NOT NULL", "valid_amount": "amount IS NOT NULL AND amount>=0"}
for name, predicate in rules.items():
    print(name, pipeline_input.filter(f"NOT ({predicate})").count())

### Solución 2

In [ ]:
if RUN_PIPELINE_CHECKS:
    prefix = f"{ident(CATALOG)}.{ident(PIPELINE_SCHEMA)}"
    display(spark.table(prefix + ".quality_observe"))
    display(spark.table(prefix + ".quality_drop"))
else:
    print("Alternativa local:", pipeline_input.count(), good.count())
# En el pipeline: dataset → Data quality → regla valid_amount; revisar passed/failed.

### Solución 3

In [ ]:
# Pasos de solución ejecutable en la UI:
# 1. Adjuntar quality_fail.sql al mismo pipeline.
# 2. Update; localizar quality_fail en estado FAILED y la regla valid_amount.
# 3. Los otros flujos pueden haber confirmado resultados.
# Simulación local del contrato, con fallo capturado para poder continuar:
try:
    assert bad.count() == 0, "valid_amount: lote rechazado"
except AssertionError as exc:
    print("Fallo esperado:", exc)

### Solución 4

In [ ]:
spark.table("pipeline_input").withColumn("amount",F.when(F.col("amount")<0,F.lit(120.0)).otherwise(F.col("amount"))).write.format("delta").mode("overwrite").saveAsTable("pipeline_input_corrected")
print("Cambia lab.source_table a", f"{CATALOG}.{SCHEMA}.pipeline_input_corrected")
print("Ejecuta Full refresh del pipeline de prácticas; verifica que quality_fail tiene 12 filas.")
assert spark.table("pipeline_input_corrected").filter("amount<0 OR amount IS NULL").count()==0

### Solución 5

In [ ]:
total, rejected = pipeline_input.count(), bad.count()
rate = rejected / total if total else 0.0
display(spark.createDataFrame([(total,rejected,rate,rate<=0.05)], "total LONG, rejected LONG, rejection_rate DOUBLE, accepted BOOLEAN"))
assert rate > 0.05
# En un job, una validación de este contrato puede fallar una tarea previa a publicación.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
Con EXPECT sin acción adicional, ¿qué pasa con una fila inválida?

A. Se borra necesariamente

B. Se conserva y se mide la violación

C. Se sustituye por cero

D. Se convierte en checkpoint

### Pregunta 2
¿Qué política descarta la fila inválida?

A. FAIL UPDATE

B. GRANT

C. ON VIOLATION DROP ROW

D. OPTIMIZE

### Pregunta 3
¿Qué debe asumirse sobre FAIL UPDATE en un pipeline con varios flujos?

A. Falla el flujo afectado; no garantiza rollback global

B. Deshace todas las tablas del catálogo

C. Solo emite un warning

D. Corrige los valores automáticamente

### Respuestas y explicación
**1. B** — La política permite observar calidad sin descartar.

**2. C** — El flujo continúa con las filas aceptadas.

**3. A** — Cada flujo puede tener su propio progreso y commits.

### Documentación oficial
- [Expectations](https://docs.databricks.com/aws/en/ldp/expectations)

## PARTE 6 - RETO FINAL
Implementa calidad por clave, importe y fecha, con cuarentena por motivo y métrica Gold de rechazo. Introduce un lote malo, recupera el pipeline y conserva evidencia del fallo.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
